In [1]:
# =============================================================================
# ETA Prediction — Step 9: AI Application (Inference Pipeline + Demo App)
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

print("Step 9 Started - AI Application")

Mounted at /content/drive
Step 9 Started - AI Application


In [2]:
# ====================== 9.1 LOAD TẤT CẢ ARTEFACTS ======================
CKPT_DIR    = "/content/drive/MyDrive/LaDe/Checkpoints_v3"   # data + feature engineering artefacts
CKPT_DIR_V4 = "/content/drive/MyDrive/LaDe/Checkpoints_v4"   # tuned model (Step 8)

required_files = {
    "final_features": f"{CKPT_DIR}/final_features.pkl",
    "te_maps":        f"{CKPT_DIR}/te_maps.pkl",
    "courier_stats":  f"{CKPT_DIR}/courier_stats.pkl",
    "coord_scaler":   f"{CKPT_DIR}/coord_scaler.pkl",
    "kmeans":         f"{CKPT_DIR}/kmeans.pkl",
    "cluster_te":     f"{CKPT_DIR}/cluster_te.pkl",
    "best_model_name": f"{CKPT_DIR_V4}/best_model_name.pkl",
}

missing = [k for k, p in required_files.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Thiếu artefact: {missing}.\n"
        "Nếu là coord_scaler / kmeans / cluster_te: quay lại Step 5, thêm joblib.dump "
        "cho 3 object này ở cuối cell 5.8 rồi chạy lại Step 5."
    )

final_features = joblib.load(required_files["final_features"])
te_maps        = joblib.load(required_files["te_maps"])
courier_stats  = joblib.load(required_files["courier_stats"])
coord_scaler   = joblib.load(required_files["coord_scaler"])
kmeans         = joblib.load(required_files["kmeans"])
cluster_te     = joblib.load(required_files["cluster_te"])
best_model_name = joblib.load(required_files["best_model_name"])

if best_model_name == "LightGBM":
    model = joblib.load(f"{CKPT_DIR_V4}/lgb_model_tuned.pkl")
    model_kind = "lightgbm"
else:
    model = joblib.load(f"{CKPT_DIR_V4}/xgb_model_tuned.pkl")
    model_kind = "xgboost"

# Global fallback values (giống Step 5, cần cho courier cold-start / category lạ)
global_mean_log = pd.Series(te_maps[list(te_maps.keys())[0]]).mean()  # xấp xỉ, chỉ dùng làm fallback cuối cùng
global_eta_mean  = courier_stats['courier_eta_mean'].mean()
global_eta_std   = courier_stats['courier_eta_std'].mean()
global_dist_mean = courier_stats['courier_dist_mean'].mean()

print(f"Model đang dùng: {best_model_name} ({model_kind})")
print(f"Số feature: {len(final_features)}")

Model đang dùng: XGBoost (xgboost)
Số feature: 24


In [3]:
# ====================== 9.2 HÀM HAVERSINE (giống Features_Engineer) ======================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

In [4]:
# ====================== 9.3 ETAPredictor — REPLICATE FEATURE ENGINEERING CHO 1 ĐƠN HÀNG ======================
class ETAPredictor:
    """
    Bọc toàn bộ pipeline (feature engineering + model) thành 1 class duy nhất
    để dùng cho inference, tránh lặp code và tránh train/serve skew.

    Input mong đợi (dict), tương tự 1 dòng raw trong pickup_yt.csv:
        {
            'lat': float, 'lng': float,
            'accept_gps_lat': float hoặc None,
            'accept_gps_lng': float hoặc None,
            'pickup_gps_lat': float hoặc None,   # thường KHÔNG có ở thời điểm dự đoán thật
            'pickup_gps_lng': float hoặc None,
            'accept_time': str 'YYYY-MM-DD HH:MM:SS' hoặc pd.Timestamp,
            'time_window_start': str/Timestamp,
            'time_window_end': str/Timestamp,
            'region_id': any, 'aoi_id': any, 'aoi_type': any,
            'courier_id': any,
        }

    Lưu ý quan trọng: ở thời điểm dự đoán thật (lúc vừa accept đơn), 'pickup_gps'
    thường CHƯA tồn tại (đó chính là target ta đang dự đoán thời điểm pickup).
    Vì vậy hàm này coi distance_km là khoảng cách từ courier hiện tại (lat/lng)
    đến accept_gps (nếu có) — đúng với ý nghĩa "quãng đường còn lại cần đi".
    """

    def __init__(self, model, model_kind, final_features, te_maps,
                 courier_stats, coord_scaler, kmeans, cluster_te,
                 global_eta_mean, global_eta_std, global_dist_mean):
        self.model = model
        self.model_kind = model_kind
        self.final_features = final_features
        self.te_maps = te_maps
        self.courier_stats = courier_stats.set_index('courier_id')
        self.coord_scaler = coord_scaler
        self.kmeans = kmeans
        self.cluster_te = cluster_te
        self.global_eta_mean = global_eta_mean
        self.global_eta_std = global_eta_std
        self.global_dist_mean = global_dist_mean
        # global_mean cho target encoding fallback: trung bình toàn bộ giá trị map
        self.global_te_mean = {col: float(np.mean(list(m.values))) for col, m in te_maps.items()}

    def _engineer(self, order: dict) -> pd.DataFrame:
        accept_time = pd.to_datetime(order['accept_time'])
        window_start = pd.to_datetime(order['time_window_start'])
        window_end = pd.to_datetime(order['time_window_end'])

        lat, lng = order['lat'], order['lng']
        accept_gps_lat = order.get('accept_gps_lat')
        accept_gps_lng = order.get('accept_gps_lng')

        if accept_gps_lat is not None and accept_gps_lng is not None:
            distance_km = haversine(lat, lng, accept_gps_lat, accept_gps_lng)
        else:
            # fallback giống Features_Engineer Section 3: dùng chính lat/lng -> distance 0
            # (đây chính là pattern lỗi cũ — chỉ chấp nhận được làm fallback cuối, không phải default)
            distance_km = 0.0

        accept_hour = accept_time.hour
        accept_minute = accept_time.minute
        day_of_week = accept_time.dayofweek
        is_weekend = int(day_of_week >= 5)
        is_morning_rush = int(7 <= accept_hour <= 9)
        is_evening_rush = int(16 <= accept_hour <= 19)
        is_long_haul = int(distance_km > 8)
        gps_both = int(accept_gps_lat is not None and accept_gps_lng is not None)

        hour_sin = np.sin(2 * np.pi * accept_hour / 24)
        hour_cos = np.cos(2 * np.pi * accept_hour / 24)

        time_to_window_start = (window_start - accept_time).total_seconds() / 60
        time_window_duration = (window_end - window_start).total_seconds() / 60

        distance_x_hour = distance_km * accept_hour

        # aoi_frequency / courier_frequency: ở production thật, 2 giá trị này nên
        # được cập nhật định kỳ từ 1 bảng thống kê (không tính lại từ raw mỗi request).
        # Ở đây dùng courier_stats đã có sẵn làm proxy đơn giản; nếu cần chính xác
        # hơn, lưu thêm 1 bảng aoi_frequency/courier_frequency riêng từ Step 5.
        aoi_frequency = order.get('aoi_frequency', 1)
        courier_frequency = order.get('courier_frequency', 1)

        courier_id = order.get('courier_id')
        if courier_id in self.courier_stats.index:
            row = self.courier_stats.loc[courier_id]
            courier_eta_mean = row['courier_eta_mean']
            courier_eta_std = row['courier_eta_std']
            courier_dist_mean = row['courier_dist_mean']
        else:
            courier_eta_mean = self.global_eta_mean
            courier_eta_std = self.global_eta_std
            courier_dist_mean = self.global_dist_mean

        coords_scaled = self.coord_scaler.transform([[lat, lng]])
        coord_cluster_v2 = int(self.kmeans.predict(coords_scaled)[0])
        coord_cluster_v2_te = self.cluster_te.get(coord_cluster_v2, np.mean(list(self.cluster_te.values)))

        def te(col, raw_value):
            mapping = self.te_maps[col]
            return mapping.get(raw_value, self.global_te_mean[col])

        row = {
            'lat': lat, 'lng': lng, 'distance_km': distance_km,
            'accept_minute': accept_minute, 'day_of_week': day_of_week,
            'hour_sin': hour_sin, 'hour_cos': hour_cos,
            'time_to_window_start': time_to_window_start,
            'time_window_duration': time_window_duration,
            'aoi_frequency': aoi_frequency, 'courier_frequency': courier_frequency,
            'distance_x_hour': distance_x_hour,
            'courier_eta_mean': courier_eta_mean,
            'courier_eta_std': courier_eta_std,
            'courier_dist_mean': courier_dist_mean,
            'region_id_te': te('region_id', order.get('region_id')),
            'aoi_id_te': te('aoi_id', order.get('aoi_id')),
            'aoi_type_te': te('aoi_type', order.get('aoi_type')),
            'is_weekend_te': te('is_weekend', is_weekend),
            'is_morning_rush_te': te('is_morning_rush', is_morning_rush),
            'is_evening_rush_te': te('is_evening_rush', is_evening_rush),
            'is_long_haul_te': te('is_long_haul', is_long_haul),
            'gps_both_te': te('gps_both', gps_both),
            'coord_cluster_v2_te': coord_cluster_v2_te,
        }

        return pd.DataFrame([row])[self.final_features]

    def predict(self, order: dict) -> float:
        X = self._engineer(order)
        if self.model_kind == "lightgbm":
            pred_log = self.model.predict(X, num_iteration=self.model.best_iteration)[0]
        else:
            pred_log = self.model.predict(X, iteration_range=(0, self.model.best_iteration + 1))[0]
        return float(np.expm1(pred_log))

    def predict_batch(self, orders: list) -> list:
        return [self.predict(o) for o in orders]

In [5]:
# ====================== 9.4 KHỞI TẠO PREDICTOR ======================
predictor = ETAPredictor(
    model=model, model_kind=model_kind,
    final_features=final_features, te_maps=te_maps,
    courier_stats=courier_stats, coord_scaler=coord_scaler,
    kmeans=kmeans, cluster_te=cluster_te,
    global_eta_mean=global_eta_mean, global_eta_std=global_eta_std,
    global_dist_mean=global_dist_mean,
)
print("ETAPredictor sẵn sàng.")

ETAPredictor sẵn sàng.


In [6]:
# ====================== 9.5 TEST NHANH VỚI 1 ĐƠN HÀNG MẪU ======================
sample_order = {
    'lat': 37.45, 'lng': 121.45,
    'accept_gps_lat': 37.46, 'accept_gps_lng': 121.46,
    'accept_time': '2024-03-15 08:45:00',
    'time_window_start': '2024-03-15 09:00:00',
    'time_window_end': '2024-03-15 11:00:00',
    'region_id': 12, 'aoi_id': 305, 'aoi_type': 2,
    'courier_id': 88421,
}

pred_minutes = predictor.predict(sample_order)
print(f"ETA dự đoán: {pred_minutes:.1f} phút")

ETA dự đoán: 66.7 phút


In [7]:
# ====================== 9.6 SANITY CHECK: SO SÁNH VỚI PIPELINE OFFLINE ======================
# Lấy ngẫu nhiên vài dòng từ X_test (đã qua pipeline offline ở Step 5) và predict
# trực tiếp bằng model — để chắc chắn ETAPredictor không lệch với pipeline gốc
# (đây không phải so sánh end-to-end từ raw, mà so sánh tầng model serving).

X_test = joblib.load(f"{CKPT_DIR}/X_test.pkl")
y_test_min = joblib.load(f"{CKPT_DIR}/y_test_min.pkl")

sample_idx = X_test.sample(5, random_state=42).index
X_sample = X_test.loc[sample_idx]

if model_kind == "lightgbm":
    pred_log_offline = model.predict(X_sample, num_iteration=model.best_iteration)
else:
    pred_log_offline = model.predict(X_sample, iteration_range=(0, model.best_iteration + 1))

pred_min_offline = np.expm1(pred_log_offline)
true_min = y_test_min.loc[sample_idx]

check_df = pd.DataFrame({
    'true_min': true_min.values,
    'pred_min_offline': pred_min_offline,
})
print(check_df.round(2))
print("\n(Đây là model serving check, không phải full ETAPredictor — vì X_test đã ở dạng feature, không phải raw order)")

   true_min  pred_min_offline
0     139.0         63.790001
1     147.0         71.570000
2      96.0         59.759998
3      17.0         56.189999
4      62.0         93.550003

(Đây là model serving check, không phải full ETAPredictor — vì X_test đã ở dạng feature, không phải raw order)


In [8]:
# ====================== 9.7 LƯU PREDICTOR BUNDLE (để deploy độc lập) ======================
DEPLOY_DIR = "/content/drive/MyDrive/LaDe/Deploy_v1"
os.makedirs(DEPLOY_DIR, exist_ok=True)

joblib.dump(predictor, f"{DEPLOY_DIR}/eta_predictor.pkl")
print(f"Đã lưu predictor bundle (đầy đủ pipeline) vào: {DEPLOY_DIR}/eta_predictor.pkl")
print("Bundle này tự chứa model + toàn bộ artefact feature engineering -> ")
print("chỉ cần load 1 file duy nhất khi triển khai, không cần load lại từng pkl riêng lẻ.")

Đã lưu predictor bundle (đầy đủ pipeline) vào: /content/drive/MyDrive/LaDe/Deploy_v1/eta_predictor.pkl
Bundle này tự chứa model + toàn bộ artefact feature engineering -> 
chỉ cần load 1 file duy nhất khi triển khai, không cần load lại từng pkl riêng lẻ.


In [9]:
# ====================== 9.8 DEMO APP (Gradio) ======================
!pip install -q gradio

import gradio as gr

def gradio_predict(lat, lng, accept_gps_lat, accept_gps_lng,
                    accept_time, window_start, window_end,
                    region_id, aoi_id, aoi_type, courier_id):
    order = {
        'lat': lat, 'lng': lng,
        'accept_gps_lat': accept_gps_lat, 'accept_gps_lng': accept_gps_lng,
        'accept_time': accept_time,
        'time_window_start': window_start,
        'time_window_end': window_end,
        'region_id': region_id, 'aoi_id': aoi_id, 'aoi_type': aoi_type,
        'courier_id': courier_id,
    }
    try:
        eta = predictor.predict(order)
        return f"{eta:.1f} phút"
    except Exception as e:
        return f"Lỗi: {e}"

demo = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Number(label="lat", value=37.45),
        gr.Number(label="lng", value=121.45),
        gr.Number(label="accept_gps_lat", value=37.46),
        gr.Number(label="accept_gps_lng", value=121.46),
        gr.Textbox(label="accept_time (YYYY-MM-DD HH:MM:SS)", value="2024-03-15 08:45:00"),
        gr.Textbox(label="time_window_start", value="2024-03-15 09:00:00"),
        gr.Textbox(label="time_window_end", value="2024-03-15 11:00:00"),
        gr.Textbox(label="region_id", value="12"),
        gr.Textbox(label="aoi_id", value="305"),
        gr.Textbox(label="aoi_type", value="2"),
        gr.Textbox(label="courier_id", value="88421"),
    ],
    outputs=gr.Textbox(label="ETA dự đoán"),
    title="LaDe ETA Predictor",
    description="Dự đoán thời gian từ lúc accept đơn đến lúc pickup (phút).",
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d52cfd63eeedbf9173.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
